# Suite PYP — The Python Patterns dialect

The patterns of Python's `match` statement, a grammar of their own (the Language Reference, 8.6.4): the dialect
`Expressions.Python.Patterns` of the Python family, its kinds mbse-programs' patterns, its `Term` of them alone, the
expressions they hold Python3's, its holder, what a pattern binds, its editions, its printing, and its evaluation, which
refuses each kind by name (mbse-basis [CAP-276: Python's type expressions and match patterns as data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-276.md)).

In [ ]:
import { readFileSync } from "node:fs";

import { Framework } from "@mbse/expressions";
import * as IEEE754 from "@mbse/expressions/IEEE754/Expressions";
import * as ISOIEC11404 from "@mbse/expressions/ISOIEC11404/Expressions";
import * as Patterns from "@mbse/expressions/Patterns/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as PythonPatterns from "@mbse/expressions/Python/Patterns";
import * as PythonRegExp from "@mbse/expressions/Python/RegExp";
import * as PythonTypes from "@mbse/expressions/Python/Types";
import { Domains as D, Standards as FS, Symbolics as SY, Terms as F } from "@mbse/expressions/Framework";
import { Builders, Errors, JSON, Validators, YAML } from "@mbse/schemas/Framework";
import { assert, equal, form_equal, map, raises, same } from "./support.js";
import { Simplifiers as Simplification } from "@mbse/expressions/Framework";
const { NotEvaluated } = Simplification;

const { NotImplementedError, ValueError } = Errors;
const [P, PV, PText] = [Python3.Expressions, Python3.Simplifiers, Python3.Text];
const [PP, PPText] = [PythonPatterns.Expressions, PythonPatterns.Text];
const [PT, PR] = [PythonTypes.Expressions, PythonRegExp.Expressions];

const store: any = new Builders.Store(); // the framework, the standard dialects Python3's complex numbers use, and the Python family
for (const install of [Framework.install, ISOIEC11404.install, IEEE754.install, Patterns.install, Python3.install,
  PythonTypes.install, PythonPatterns.install]) {
  store.transaction(install);
}
store.transaction(PythonPatterns.define); // the versions the Patterns dialect covers
const Term: any = store.registered("Expressions.Framework.Term");
const READER = new Map<string, F.Dialect>([P.DIALECT, PT.DIALECT, PP.DIALECT].map((d) => [d.union, d])); // the dialect whose union holds a term

/** The holder of the term `spec` writes, `dialect`'s own (the Patterns dialect's unless given), written in a
 * transaction. */
function held(spec: unknown, dialect: F.Dialect = PP.DIALECT): any {
  return store.transaction(() => dialect.path.split(".").reduce((at: any, part) => at[part], store.Expressions)
    .Expression((e: any) => e.root(dialect.spec(spec))));
}

const pattern = (spec: unknown): any => F.term(held(spec)); // a term of the Patterns dialect, held by its holder
const capture = (name: string) => (u: any) => u.matchas((m: any) => m.name(name)); // a capture pattern, `name`
const wildcard = (u: any) => u.matchas((m: any) => m); // `_`
const value = (spec: unknown) => (u: any) => u.matchvalue((m: any) => m.value(P.DIALECT.spec(spec))); // a value pattern, a Python3 term or a native
const sequence = (...items: unknown[]) => (u: any) => u.matchsequence((s: any) => s.patterns(items)); // an open sequence
const listed = (...items: unknown[]) => (u: any) => u.matchsequence((s: any) => s.delimiters("[]").patterns(items)); // `[...]`
const either = (...items: unknown[]) => (u: any) => u.matchor((o: any) => o.patterns(items)); // `a | b`
const named = (inner: unknown, name: string) => (u: any) => u.matchas((m: any) => m.pattern(inner).name(name)); // `inner as name`
const names = (set: ReadonlySet<string>): string[] => [...set].sort();

<a id="PYP-01"></a>
## PYP-01 · The Patterns dialect's union, its kinds' schemas in mbse-programs' shapes, the expressions they hold Python3's, and its holder: validated, linked by `HasRoot`, round-tripped through JSON and YAML

In [ ]:
// Its kinds: mbse-programs' patterns, named and shaped as it has them; its Term lists them and invalid syntax alone, an
// expression in a pattern being a kind's own subterm, of Python3's Term; it has no helper of its own, its
// names strings.
const union = (dialect: F.Dialect): string[] => store.registered(dialect.union).branches.map((b: any) => b.name); // its Term's branches
assert(same(PP.TAGS, ["matchvalue", "matchsingleton", "matchsequence", "matchmapping", "matchclass", "matchstar", "matchas",
  "matchor"]) && PP.HELPERS.length === 0);
assert(same(union(PP.DIALECT), [...PP.TAGS, "error", "missing"]) && same([...PP.DIALECT.kinds().keys()], [...PP.TAGS, "error", "missing"]));
assert(!PP.TAGS.some((tag) => union(P.DIALECT).includes(tag) || union(PT.DIALECT).includes(tag))); // no other union of the family lists them
assert(PP.DIALECT.name() === "Python Patterns" && PP.DIALECT.tag === "python_patterns");
assert(PP.DIALECT.path === "Python.Patterns" && PP.DIALECT.union === "Expressions.Python.Patterns.Term");
const [PatternsTerm, Python3Term]: any[] = [store.registered(PP.DIALECT.union), store.registered(P.DIALECT.union)];
const kinds = new Map<string, any>(PatternsTerm.branches.map((b: any) => [b.name, b.schema]));
assert(same(PP.TAGS.map((tag) => kinds.get(tag).name), ["MatchValue", "MatchSingleton", "MatchSequence", "MatchMapping", "MatchClass",
  "MatchStar", "MatchAs", "MatchOr"].map((kind) => `Expressions.Python.Patterns.${kind}`)));
assert(equal(new Map(PP.TAGS.map((tag) => [tag, [...kinds.get(tag).properties.keys()]])), map({ // each a value object, in mbse-programs' order
  matchvalue: ["value"], matchsingleton: ["value"], matchsequence: ["delimiters", "patterns"],
  matchmapping: ["keys", "patterns", "rest"], matchclass: ["cls", "patterns", "kwd_attrs", "kwd_patterns"],
  matchstar: ["name"], matchas: ["pattern", "name"], matchor: ["patterns"] })) && !PP.TAGS.some((t) => kinds.get(t).ref));
assert(kinds.get("matchvalue").properties.get("value").schema === Python3Term && kinds.get("matchclass").properties.get("cls").schema === Python3Term);
assert(kinds.get("matchmapping").properties.get("keys").schema.element === Python3Term); // a key: a literal or a value pattern's name
assert(kinds.get("matchas").properties.get("pattern").schema === PatternsTerm && ([["matchsequence", "patterns"], ["matchmapping", "patterns"],
  ["matchclass", "patterns"], ["matchclass", "kwd_patterns"], ["matchor", "patterns"]] as [string, string][]).every(
  ([tag, p]) => kinds.get(tag).properties.get(p).schema.element === PatternsTerm));
assert(equal(F._parts(kinds.get("matchmapping")), [[], ["keys", "patterns"], ["rest"]])); // two containers of terms (API 210)
assert(equal(F._parts(kinds.get("matchclass")), [["cls"], ["patterns", "kwd_patterns"], ["kwd_attrs"]]));
assert(equal(F._parts(kinds.get("matchas")), [["pattern"], [], ["name"]]) && equal(F._parts(kinds.get("matchsingleton")), [[], [], ["value"]]));
// No subterm of Python3's or of the Types dialect's is of its Term: a match is a statement, mbse-programs' `case`
// holding its pattern through the Patterns dialect's holder. Its Term is a branch of the framework's, its dialect a
// singleton.
const holds = (schema: any): boolean => [...schema.properties.values()].some((p: any) => p.schema === PatternsTerm || p.schema.element === PatternsTerm);
assert(![P.DIALECT, PT.DIALECT].some((d) => store.registered(d.union).branches.some((b: any) => holds(b.schema))));
assert(Term.branches.find((b: any) => b.name === "python_patterns").schema === PatternsTerm);
assert(store.singleton("Expressions.Python.Patterns") === store.Instances.Expressions.Python.Patterns);
// Each dialect of the family reads its kinds, and it reads theirs.
assert(P.DIALECT.kindNamed("matchas") === PP.DIALECT.kinds().get("matchas") && PP.DIALECT.kindNamed("binop") === P.DIALECT.kinds().get("binop"));
assert(same(PP.DIALECT.reads(), [P.DIALECT, PT.DIALECT, PR.DIALECT]) && same(PT.DIALECT.reads(), [P.DIALECT, PP.DIALECT, PR.DIALECT]));
// Its holder (Q2, "(a)"): a reference object refining the framework's Expression, its root narrowed to its Term; it
// validates, HasRoot links it, and it round-trips through JSON and YAML, as its root does as a value of its Term.
const Holder: any = store.registered("Expressions.Python.Patterns.Expression");
assert(Holder.ref && Holder.refines === store.registered("Expressions.Framework.Expression") && F.isExpression(Holder));
assert(Holder.properties.get("root").schema === PatternsTerm && Holder.validate().length === 0 && same([...Holder.adjacencies.keys()], ["abi"]));
const every = either( // Point(0, y=[a, *rest]) | {'k': None, **kw} | (Color.RED as c)
  (u: any) => u.matchclass((c: any) => c.cls(P.name("Point")).patterns([value(0n)]).kwd_attrs(["y"]).kwd_patterns([
    listed(capture("a"), (s: any) => s.matchstar((m: any) => m.name("rest")))])),
  (u: any) => u.matchmapping((m: any) => m.keys([P.constant("k")]).patterns([(s: any) => s.matchsingleton((v: any) => v.value("None"))]).rest("kw")),
  named(value(P.attribute(P.name("Color"), "RED")), "c"));
const [holder, linked] = store.transaction(() => {
  const holder = store.Expressions.Python.Patterns.Expression((e: any) => e.root(PP.DIALECT.spec(every)));
  const linked = store.Schemas.OfExpression((s: any) => s.name("pyp.every").expression((a: any) => a.insert((e: any) => e.expression(holder))));
  return [holder, linked];
});
assert(Validators.Validate(store)(Holder, holder).length === 0 && PP.DIALECT.validate(holder, ["Point", "Color"]).length === 0);
assert(same([...(new Builders.OfExpression.Builder(linked, false, store) as any).expression].map((entry: any) => F.same(entry.expression, holder)), [true]));
const SOURCE = "Point(0, y=[a, *rest]) | {'k': None, **kw} | (Color.RED as c)";
assert(F.dialectOf(F.term(holder)) === PP.DIALECT.union && PPText.ToText(holder) === SOURCE);
for (const [write, read] of [[JSON.ToJSON(store), JSON.FromJSON(store)], [YAML.ToYAML(store), YAML.FromYAML(store)]] as [any, any][]) {
  assert(F.same(read(Holder, write(Holder, holder)).root, holder.root));
  assert(F.same(read(PatternsTerm, write(PatternsTerm, F.term(holder))), F.term(holder)));
}
// The schema refuses a pattern where a union that lacks it is held: Python3's holder's root.
raises(Error, () => store.transaction(() => store.Expressions.Python.Python3.Expression(
  (e: any) => e.root(PP.DIALECT.make(new F.Form("matchas", {}, [null]))))), "'matchas' is not a member of a union value");
// Another family's term is refused, as before.
raises(ValueError, () => PP.DIALECT.kind(F.term(held(Patterns.Expressions.let_("a", 1n, Patterns.Expressions.variable("a")),
  Patterns.Expressions.DIALECT))), "a Patterns let is not a Python Patterns term");

<a id="PYP-02"></a>
## PYP-02 · Each kind and its checks: mbse-programs' (a choice among a few, a pattern for each key and keyword, two alternatives) and `ast`'s, its capture names; forms of two containers and `make`, and inference

In [ ]:
// Each kind's checks: mbse-programs' (a singleton and delimiters among a few, a mapping's pattern for each key, a
// class's pattern for each keyword, an `or` of two patterns), `ast`'s (an `as` pattern with a pattern has a name), and
// its names' (an identifier; a capture name not `_`, which captures nothing). A value pattern's value, a mapping's keys
// and a class's class are Python3's terms, their names references the scope must bind.
const validate = (spec: unknown, bound: string[] = []): string[] => PP.DIALECT.validate(pattern(spec), bound);
const singleton = (text: string) => (u: any) => u.matchsingleton((v: any) => v.value(text));
const mapping = (keys: unknown[], patterns: unknown[], rest: string | null = null) => (u: any) => u.matchmapping(
  (m: any) => (rest === null ? m.keys(keys).patterns(patterns) : m.keys(keys).patterns(patterns).rest(rest)));
for (const [spec, problems] of [
  [value(1n), []], [value(P.unaryop("-", 1n)), []], [value(P.attribute(P.name("Color"), "RED")), ["value: value: variable 'Color' is not bound"]],
  [(u: any) => u.matchvalue((m: any) => m), ["a matchvalue needs a value"]],
  [singleton("None"), []], [singleton("True"), []], [singleton("False"), []],
  [singleton("Maybe"), ["MatchSingleton.value cannot be 'Maybe'"]],
  [(u: any) => u.matchsingleton((v: any) => v), ["a matchsingleton needs a value"]],
  [sequence(), []], [listed(capture("a"), wildcard), []],
  [(u: any) => u.matchsequence((s: any) => s.delimiters("()").patterns([capture("a")])), []],
  [(u: any) => u.matchsequence((s: any) => s.delimiters("{}")), ["MatchSequence.delimiters cannot be '{}'"]],
  [sequence(capture("1a")), ["argument 0: a matchas's name must be an identifier, got '1a'"]],
  [mapping([P.constant("k")], [capture("v")], "rest"), []], [mapping([], [], "rest"), []],
  [mapping([P.constant("k"), P.constant("j")], [capture("v")]), ["a MatchMapping needs a pattern for each key"]],
  [mapping([P.constant("k")], [capture("v")], "1r"), ["a matchmapping's rest must be an identifier, got '1r'"]],
  [mapping([P.constant("k")], [capture("v")], "_"), ["a matchmapping's rest cannot be '_'"]],
  [mapping([P.name("k")], [capture("1v")]), ["keys 0: variable 'k' is not bound", "patterns 0: a matchas's name must be an identifier, got '1v'"]],
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("int")).patterns([capture("n")])), []], // `int(n)`
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("int")).kwd_attrs(["real", "imag"]).kwd_patterns([wildcard, value(0n)])), []],
  [(u: any) => u.matchclass((c: any) => c.patterns([wildcard])), ["a matchclass needs a cls"]],
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("int")).kwd_attrs(["real"])), ["a MatchClass needs a pattern for each keyword"]],
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("int")).kwd_attrs(["re al"]).kwd_patterns([wildcard])),
    ["a matchclass's kwd_attr must be an identifier, got 're al'"]],
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("int")).kwd_patterns([wildcard])), ["a MatchClass needs a pattern for each keyword"]],
  [(u: any) => u.matchstar((s: any) => s.name("rest")), []], [(u: any) => u.matchstar((s: any) => s), []], // `*_`
  [(u: any) => u.matchstar((s: any) => s.name("_")), ["a matchstar's name cannot be '_'"]],
  [(u: any) => u.matchstar((s: any) => s.name("a b")), ["a matchstar's name must be an identifier, got 'a b'"]],
  [wildcard, []], [capture("x"), []], [named(listed(), "x"), []],
  [(u: any) => u.matchas((m: any) => m.pattern(capture("x"))), ["a matchas with a pattern needs a name"]],
  [capture("_"), ["a matchas's name cannot be '_'"]],
  [named(capture("1x"), "y"), ["pattern: a matchas's name must be an identifier, got '1x'"]],
  [either(value(1n), value(2n)), []], [either(value(1n)), ["a MatchOr needs two patterns"]],
] as [unknown, string[]][]) {
  assert(same(validate(spec), problems), `${problems} ${validate(spec)}`);
}
assert(validate(value(P.attribute(P.name("Color"), "RED")), ["Color"]).length === 0);
assert(PP.problems(F.node("matchas", F.kindOf(pattern(wildcard))[1])).length === 0);
// Each kind an application of no domain known statically, which the framework reads by its structure alone: its form,
// from which make writes it again, by the dialect whose union holds it; a mapping's and a class's two containers of
// terms are each given their size by a part named by the container (API 210), and read by container.
assert(PP.TAGS.every((tag) => PP.DIALECT.kinds().get(tag)!.role === F.APPLICATION));
const tree = pattern(every);
const walked = [...F.walk(tree)];
assert(same(names(new Set(walked.filter((node) => F.dialectOf(node) === PP.DIALECT.union).map((node) => F.kindOf(node)[0]))), [...PP.TAGS].sort()));
for (const node of walked) {
  const dialect = READER.get(F.dialectOf(node))!;
  const made = F.term(held(dialect.make(F.form(node)), dialect));
  assert(F.same(made, node) && made !== node);
}
const pairs = pattern(mapping([P.constant("k"), P.constant("j")], [capture("v"), wildcard], "kw"));
const pairsForm = F.form(pairs);
assert(same([...pairsForm.attributes.keys()], ["rest", "keys", "patterns"]) && pairsForm.attributes.get("rest") === "kw");
assert((["keys", "patterns"] as const).every((name) => {
  const part = pairsForm.attributes.get(name) as F.Part;
  return part.kind === name && part.attributes.size === 0 && part.size === 2;
}));
const pairsNode = F.node("matchmapping", F.kindOf(pairs)[1]);
assert(same([...pairsNode.lists.keys()], ["keys", "patterns"]) && same([...pairsNode.lists.values()].map((held_) => held_.length), [2, 2]) && pairsNode.many.length === 4);
const point = pattern((u: any) => u.matchclass((c: any) => c.cls(P.name("Point")).patterns([capture("x")]).kwd_attrs(["y", "z"]).kwd_patterns([wildcard, value(0n)])));
const formed = F.form(point);
assert(same((formed.attributes.get("kwd_attrs") as any).values, ["y", "z"]) && formed.arguments.length === 4); // its class, then its containers
assert((formed.attributes.get("patterns") as F.Part).size === 1 && (formed.attributes.get("kwd_patterns") as F.Part).size === 2);
assert(form_equal(F.form(pattern(wildcard)), new F.Form("matchas", {}, [null])) && F.nameOf(pattern(capture("x"))) === "x");
assert(PP.DIALECT.infer(tree, { Point: D.Anything, Color: D.Anything }) === D.Anything);

<a id="PYP-03"></a>
## PYP-03 · What a pattern binds: its capture names, wherever they nest, which the `case` holding it binds; none of its kinds a binder

In [ ]:
// What a pattern binds (`boundNames`): its capture names, an `as` pattern's and a star pattern's name and a mapping's
// rest, wherever its sequence, mapping, class, `as` and `or` patterns nest them; a value or a singleton pattern, the
// wildcard, `*_`, an expression a pattern holds (whose names are references) and invalid syntax bind none.
const binds = (spec: unknown): string[] => names(PP.boundNames(pattern(spec)));
assert(same(binds(capture("a")), ["a"]) && binds(wildcard).length === 0 && binds(value(P.name("a"))).length === 0);
assert(binds(singleton("None")).length === 0 && binds((u: any) => u.matchstar((s: any) => s)).length === 0);
assert(same(binds(named(listed(capture("a"), (s: any) => s.matchstar((m: any) => m.name("b"))), "c")), ["a", "b", "c"]));
assert(same(binds(mapping([P.name("k")], [capture("v")], "rest")), ["rest", "v"])); // not its key's name, a reference
assert(same(binds((u: any) => u.matchclass((c: any) => c.cls(P.name("C")).patterns([capture("a")]).kwd_attrs(["x"]).kwd_patterns(
  [capture("b")]))), ["a", "b"])); // not its class's name, nor its keyword's
assert(same(binds(either(capture("a"), named(value(1n), "a"))), ["a"]));
assert(binds(sequence((u: any) => u.error((e: any) => e.text("a +").terms([PP.DIALECT.spec(capture("a"))])))).length === 0);
// A pattern binds, but is no binder of its own: none of its kinds refines Let, and none has scopes; the `case` that holds
// it, mbse-programs' statement, binds its names in its guard and its body. Its expressions' names are free.
assert(!PP.TAGS.some((tag) => F.isBinder(kinds.get(tag)) || PP.DIALECT.kinds().get(tag)!.scopes !== null));
assert(same(names(SY.free(pattern(every), PP.DIALECT)), ["Color", "Point"]));
assert(same(names(SY.free(pattern(mapping([P.name("k")], [capture("v")])), PP.DIALECT)), ["k"]));

<a id="PYP-04"></a>
## PYP-04 · A tree mixing patterns with Python3's expressions, validated, inferred and printed from the Patterns dialect's holder, and read by each dialect of the family

In [ ]:
// A tree mixing patterns with Python3's expressions: a value pattern's dotted name and literals (a negative number, a
// complex one), a mapping's keys, a class's class, each Python3's term the pattern's own subterm; validated, inferred and
// printed from the Patterns dialect's holder; every dialect of the family reads it.
const mixed = either(
  (u: any) => u.matchclass((c: any) => c.cls(P.attribute(P.name("shapes"), "Point")).kwd_attrs(["x", "y"]).kwd_patterns([
    value(P.unaryop("-", 1n)), named(either(value(0n), value(P.constant(new P.Complex(0, 2)))), "y")])),
  (u: any) => u.matchmapping((m: any) => m.keys([P.constant("kind"), P.attribute(P.name("Keys"), "SIZE")]).patterns([
    value("circle"), sequence(capture("w"), capture("h"))])));
const SOURCE_MIXED = "shapes.Point(x=-1, y=0 | 2j as y) | {'kind': 'circle', Keys.SIZE: (w, h)}";
const own = pattern(mixed);
assert(PPText.ToText(own) === SOURCE_MIXED && PText.ToText(own) === SOURCE_MIXED && PPText.ToText === PText.ToText);
assert(same(PP.DIALECT.validate(own), ["argument 0: cls: value: variable 'shapes' is not bound",
  "argument 1: keys 1: value: variable 'Keys' is not bound"]));
assert([PP.DIALECT, P.DIALECT, PT.DIALECT].every((dialect) => dialect.validate(own, ["shapes", "Keys"]).length === 0));
assert(same(names(SY.free(own, PP.DIALECT)), ["Keys", "shapes"]) && same(names(PP.boundNames(own)), ["h", "w", "y"]));
assert(PP.DIALECT.infer(own, { shapes: D.Anything, Keys: D.Anything }) === D.Anything);
const cls = F.form(F.form(own).arguments[0]).arguments[0]; // the class pattern's class: Python3's term
assert(F.dialectOf(cls) === P.DIALECT.union && F.dialectOf(F.form(own).arguments[0]) === PP.DIALECT.union);

<a id="PYP-05"></a>
## PYP-05 · Invalid syntax where a pattern stands: validated, printed as written, and left in the residual by simplification, named

In [ ]:
// Invalid syntax: the Patterns dialect's Term holds `error` and `missing`, where a pattern stands; validation reports
// them, printers write them as written, and evaluation refuses them by the Patterns dialect's name.
const broken = pattern(listed((i: any) => i.error((e: any) => e.text("a +").terms([PP.DIALECT.spec(capture("a"))])),
  (i: any) => i.missing((m: any) => m.kind("pattern"))));
assert(same(PP.DIALECT.validate(broken), ["argument 0: invalid syntax 'a +'", "argument 1: missing pattern"]));
assert(PText.ToText(broken) === "[a +, ]" && PP.boundNames(broken).size === 0);
assert(same(F.form(broken).arguments.map((a) => PP.DIALECT.kind(a)[2].role), [F.INVALID, F.INVALID]));
const asMissing = pattern((u: any) => u.matchas((m: any) => m.pattern((p: any) => p.missing((r: any) => r)).name("x")));
assert(same(PP.DIALECT.validate(asMissing), ["pattern: missing a term"]) && PText.ToText(asMissing) === " as x");
raises(ValueError, () => PText.ToText(pattern(either((e: any) => e.error((r: any) => r), wildcard))), "Python Patterns error has no text to print");
raises(NotEvaluated, () => PV.OfAny(pattern((u: any) => u.missing((m: any) => m))).evaluate(), "Python Patterns missing is not simplified");

<a id="PYP-06"></a>
## PYP-06 · Its editions as data: Python 3.10 to 3.15, each with every one of its own kinds

In [ ]:
// Its editions, as data: its define covers Python 3.10 to 3.15, owned by their published names, each with every one of
// its own kinds (all of them since Python 3.10, as mbse-programs' SINCE has them) and none of the kinds it reads; run
// again, it writes nothing.
const patterns_ = store.Instances.Expressions.Python.Patterns;
const VERSIONS = Array.from({ length: 6 }, (_, i) => `Python 3.${10 + i}`);
assert(same([...FS.standards(patterns_).keys()], VERSIONS) && equal(PP.SINCE, new Map(PP.TAGS.map((tag) => [tag, 10]))));
assert(same(FS.tags(FS.find(patterns_, "Python 3.10")!), ["Python 3.10"]) && FS.find(patterns_, "Python 3.9") === null);
assert(equal(FS.properties(FS.find(patterns_, "Python 3.15")!), map({ bodies: ["org.python"], revision: ["version", "3.15"] })));
assert([...FS.standards(patterns_).values()].every((edition) => same(FS.kinds(edition), PP.TAGS.map((tag) => kinds.get(tag)))));
assert(PP.TAGS.every((tag) => same([...FS.of(patterns_, kinds.get(tag)).keys()], VERSIONS)));
assert(FS.of(patterns_, Python3Term.branches[0].schema).size === 0 && FS.of(patterns_, store.registered("Expressions.Framework.Error")).size === 0);
assert(store.transaction(PythonPatterns.define)[0].get("Python_3_10") === FS.find(patterns_, "Python 3.10"));
assert(same([...FS.standards(patterns_).keys()], VERSIONS)); // nothing written again

<a id="PYP-07"></a>
## PYP-07 · Printing every kind as mbse-programs writes patterns, parenthesized only where a pattern needs it

In [ ]:
// Printing, as mbse-programs' printer writes patterns (`p`): a value pattern as the expression it is, a singleton, a
// sequence in its delimiters or open (as a `case` writes it, at the root), a mapping, a class, a star, an `as` pattern,
// a capture and the wildcard, and an `or` pattern; an open sequence parenthesized where a pattern nests it, an `as`
// pattern where an `as` or an `or` pattern holds it, and an `or` pattern where an `or` pattern holds it.
const printed: [unknown, string][] = [
  [value(1n), "1"], [value(P.unaryop("-", 1n)), "-1"], [value("s"), "'s'"], [value(P.attribute(P.name("Color"), "RED")), "Color.RED"],
  [singleton("None"), "None"], [singleton("True"), "True"],
  [sequence(capture("a"), capture("b")), "a, b"], [sequence(capture("a")), "a,"], [sequence(), "()"],
  [listed(capture("a"), (s: any) => s.matchstar((m: any) => m.name("rest"))), "[a, *rest]"], [listed(), "[]"],
  [(u: any) => u.matchsequence((s: any) => s.delimiters("()").patterns([capture("a")])), "(a,)"],
  [(u: any) => u.matchsequence((s: any) => s.delimiters("()").patterns([capture("a"), capture("b")])), "(a, b)"],
  [listed(capture("a"), sequence(capture("b"), capture("c"))), "[a, (b, c)]"],
  [listed(named(capture("a"), "b"), either(value(1n), value(2n))), "[a as b, 1 | 2]"],
  [mapping([], []), "{}"], [mapping([P.constant("k"), P.constant(1n)], [capture("v"), wildcard], "rest"), "{'k': v, 1: _, **rest}"],
  [(u: any) => u.matchclass((c: any) => c.cls(P.name("C"))), "C()"],
  [(u: any) => u.matchclass((c: any) => c.cls(P.attribute(P.name("m"), "C")).patterns([capture("a"), sequence(capture("b"), capture("c"))])
    .kwd_attrs(["x"]).kwd_patterns([either(value(1n), value(2n))])), "m.C(a, (b, c), x=1 | 2)"],
  [(u: any) => u.matchstar((s: any) => s.name("rest")), "*rest"], [(u: any) => u.matchstar((s: any) => s), "*_"],
  [wildcard, "_"], [capture("x"), "x"], [named(listed(capture("a")), "x"), "[a] as x"],
  [named(named(capture("a"), "b"), "c"), "(a as b) as c"], [named(either(value(1n), value(2n)), "c"), "1 | 2 as c"],
  [named(sequence(capture("a"), capture("b")), "c"), "(a, b) as c"],
  [either(capture("a"), wildcard), "a | _"], [either(either(value(1n), value(2n)), value(3n)), "(1 | 2) | 3"],
  [either(named(capture("a"), "b"), sequence(capture("c"), capture("d"))), "(a as b) | (c, d)"],
];
for (const [spec, source] of printed) assert(PText.ToText(pattern(spec)) === source, `${source} ${PText.ToText(pattern(spec))}`);
const printedKinds = new Set(printed.map(([spec]) => F.kindOf(pattern(spec))[0]));
assert(PP.TAGS.every((tag) => printedKinds.has(tag))); // every kind
assert(same([...PPText.TEXTS.keys()], [...PP.TAGS]) && PPText.OPEN < PPText.AS && PPText.AS < PPText.OR && PPText.OR < PPText.CLOSED);

<a id="PYP-08"></a>
## PYP-08 · Simplification leaves each of its kinds in the residual, named by the Patterns dialect

In [ ]:
// Evaluation (Q11): Python3's evaluator evaluates none of its kinds, and refuses each by the Patterns dialect's name; a
// pattern is matched by mbse-programs' `match` statement, which is not an expression.
for (const [spec, tag] of [[value(1n), "matchvalue"], [singleton("None"), "matchsingleton"], [sequence(), "matchsequence"],
  [mapping([], []), "matchmapping"], [(u: any) => u.matchclass((c: any) => c.cls(P.name("C"))), "matchclass"],
  [(u: any) => u.matchstar((s: any) => s), "matchstar"], [wildcard, "matchas"], [either(wildcard, wildcard), "matchor"]] as [unknown, string][]) {
  raises(NotEvaluated, () => PV.OfAny(pattern(spec)).evaluate(), `Python Patterns ${tag} is not simplified`);
}

<a id="PYP-09"></a>
## PYP-09 · Completeness: each of `ast`'s pattern nodes, kept as data, is one of its kinds, and each of its kinds one of them; a `case` is a statement's part

In [ ]:
// Completeness (Q9): `grammars/python3-ast.json` lists `ast`'s pattern nodes (class `pattern`, Python 3.10 on), each
// mapped to one of the Patterns dialect's kinds, and every kind is one of them; `match_case`, a statement's part, is
// listed as not an expression (DIA-20 checks the file whole, and, in Python, against the running Python's `ast`).
const grammar = globalThis.JSON.parse(readFileSync("../../grammars/python3-ast.json", "utf8"));
const entries = grammar.nodes.filter((node: any) => node.class === "pattern");
assert(equal(entries.map((node: any) => [node.node, node.since, node.kinds]), ["MatchValue", "MatchSingleton", "MatchSequence",
  "MatchMapping", "MatchClass", "MatchStar", "MatchAs", "MatchOr"].map((kind) => [kind, "3.10", [kind.toLowerCase()]])));
assert(same([...new Set(entries.flatMap((node: any) => node.kinds))].sort(), [...PP.TAGS].sort()));
const matchCase = grammar.nodes.find((node: any) => node.node === "match_case");
assert(matchCase.class === "helper" && matchCase.since === "3.10" && "not" in matchCase);